# Statistical Models

In this notebook, I apply statistical forecasting models to the cleaned sales time series from Week 1.
The goal is to train statistical models on data from **2013-01-01 to 2013-12-31** and evaluate their forecasts on data from **2014-01-01 to 2014-03-31**.

# 1. Environment Setup

In [ ]:
# pandas - for working with tabular data
import pandas as pd

# numpy - for numerical operations
import numpy as np

# matplotlib - for plotting charts
import matplotlib.pyplot as plt

# seaborn - for improved chart styling
import seaborn as sns

# statsmodels - for time series analysis and forecasting
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# scikit-learn - for evaluation metrics
from sklearn.metrics import mean_absolute_error, r2_score

# warnings - to hide unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

# 2. Data Loading

In [ ]:
# Load the cleaned main time series from Week 1
timeseries = pd.read_csv("../../data/timeseries_cleaned.csv")

In [ ]:
# Display the first rows of the cleaned time series
timeseries.head()

In [ ]:
timeseries.shape

# 3. Data Preparation and Exploration

In [ ]:
# Check basic information about the cleaned time series
timeseries.info()

In [ ]:
# Convert the date column to datetime format
timeseries["date"] = pd.to_datetime(timeseries["date"])

In [ ]:
# Check data types after datetime conversion
print(timeseries.dtypes)

In [ ]:
# Check the date range of the cleaned time series
print("Date range:", timeseries["date"].min(), "to", timeseries["date"].max())

In [ ]:
# Set the date column as the index for time series modeling
timeseries = timeseries.set_index("date")

timeseries.head()

In [ ]:
# Plot the cleaned sales time series
plt.figure(figsize=(14, 5))
plt.plot(timeseries.index, timeseries["unit_sales"])
plt.title("Cleaned Sales Time Series")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.show()

# 4. Time Series Decomposition

In [ ]:
# Decompose the time series into trend, seasonality, and residuals
decomposition = seasonal_decompose(timeseries["unit_sales"], model="additive", period=7)

# Plot the decomposition components
decomposition.plot()
plt.show()

# 5. Stationarity Check

In [ ]:
# Perform the Augmented Dickey-Fuller test
adf_result = adfuller(timeseries["unit_sales"])

print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"   {key}: {value}")

The ADF test suggests that the time series is likely stationary, because the p-value is below 0.05. This means we can proceed with statistical modeling without applying an additional differencing step at this stage.

# 6. Autocorrelation Analysis

In [ ]:
# Plot the autocorrelation function (ACF)
plt.figure(figsize=(10, 4))
plot_acf(timeseries["unit_sales"], lags=30)
plt.show()

In [ ]:
# Plot the partial autocorrelation function (PACF)
plt.figure(figsize=(10, 4))
plot_pacf(timeseries["unit_sales"], lags=30)
plt.show()

The ACF and PACF plots show several significant spikes, including a visible repeating weekly pattern. This suggests that autoregressive and seasonal components may be useful in the statistical forecasting models.

# 7. Exploratory Analysis for Statistical Modelling

This section explores a few additional patterns in the sales series that may help interpret the behavior of the data before fitting statistical forecasting models.

In [ ]:
# Plot the distribution and boxplot of unit sales
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
plt.hist(timeseries["unit_sales"], bins=30, edgecolor="black")
plt.title("Distribution of Unit Sales")
plt.xlabel("Unit Sales")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
plt.boxplot(timeseries["unit_sales"])
plt.title("Boxplot of Unit Sales")
plt.ylabel("Unit Sales")

plt.tight_layout()
plt.show()

The sales distribution is right-skewed, with most observations concentrated in the mid-range and a smaller number of high-sales days. The boxplot also shows several upper outliers, which suggests that occasional sales spikes are present in the series.

In [ ]:
# Create a day-of-week column for exploratory analysis
timeseries["day_of_week"] = timeseries.index.dayofweek

# Calculate average sales by day of week
avg_sales_by_day = timeseries.groupby("day_of_week")["unit_sales"].mean()

# Define day labels and colors
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
bar_colors = ["steelblue", "steelblue", "steelblue", "steelblue", "steelblue", "darkorange", "darkorange"]

# Plot average sales by day of week
plt.figure(figsize=(10, 4))
plt.bar(day_labels, avg_sales_by_day.values, color=bar_colors)
plt.title("Average Unit Sales by Day of Week (orange = weekend)")
plt.xlabel("")
plt.ylabel("Average Unit Sales")
plt.grid(axis="y", alpha=0.3)
plt.show()

Average sales vary noticeably across the week. The highest values appear on Saturday and Sunday, which suggests a strong weekend effect in the sales series.

Next, I check whether oil prices show a visible relationship with sales over time.

In [ ]:
# Load the processed oil dataset for exploratory analysis
oil = pd.read_csv("../../data/processed/oil_interpolated.csv")

# Convert the oil date column to datetime
oil["date"] = pd.to_datetime(oil["date"])

In [ ]:
# Merge the sales series with oil prices for exploratory analysis
sales_oil = timeseries.reset_index().merge(
    oil[["date", "oil_price_interpolated"]],
    on="date",
    how="left"
)

# Fill missing oil values after the merge
sales_oil["oil_price_interpolated"] = sales_oil["oil_price_interpolated"].ffill()

In [ ]:
# Plot the relationship between oil prices and unit sales
plt.figure(figsize=(8, 5))
plt.scatter(
    sales_oil["oil_price_interpolated"],
    sales_oil["unit_sales"],
    alpha=0.5
)
plt.title("Relationship Between Oil Prices and Unit Sales")
plt.xlabel("Oil Price")
plt.ylabel("Unit Sales")
plt.show()

The scatter plot does not show a strong linear relationship between oil prices and unit sales. Any effect of oil prices on sales is likely to be weak, indirect, or mixed with other factors.

In [ ]:
# Check correlations between numeric variables
sales_oil[["unit_sales", "oil_price_interpolated"]].corr()

# 8. Train-Test Split

In [ ]:
# Split the time series into train and test sets using the official project periods
train = timeseries.loc["2013-01-01":"2013-12-31"].copy()
test = timeseries.loc["2014-01-01":"2014-03-31"].copy()

In [ ]:
# Check the shape and date range of the train and test sets
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("Train period:", train.index.min(), "to", train.index.max())
print("Test period:", test.index.min(), "to", test.index.max())

The test period contains 89 days, which matches the project evaluation window

In [ ]:
# Visualize the train-test split
plt.figure(figsize=(14, 4))
plt.plot(train.index, train["unit_sales"], label="Training Data", color="steelblue", linewidth=1.2)
plt.plot(test.index, test["unit_sales"], label="Test Data", color="darkorange", linewidth=1.2)
plt.axvline(x=test.index.min(), color="red", linestyle="--", linewidth=1.5, label="Split Point")
plt.title("Train / Test Split")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 9. Check Stationarity on the Training Series

In [ ]:
# Define a helper function to check stationarity with the ADF test
def check_stationarity(series, name):
    result = adfuller(series.dropna())
    p_value = result[1]

    print("ADF Test —", name)
    print("  p-value:", round(p_value, 4))

    if p_value <= 0.05:
        print("  Result: Stationary — ready for ARIMA/SARIMAX")
    else:
        print("  Result: Not stationary — differencing may be needed")

    print()

# Test the training sales series
check_stationarity(train["unit_sales"], "unit_sales (training)")

In [ ]:
# Plot ACF and PACF for the training series with every lag shown on the x-axis
series = train["unit_sales"].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# ACF
plot_acf(series, lags=30, ax=axes[0])
axes[0].set_title("ACF — Training Series")
axes[0].set_xlabel("Lags (days back)")
axes[0].set_xticks(range(0, 31, 1))

# PACF
plot_pacf(series, lags=30, ax=axes[1])
axes[1].set_title("PACF — Training Series")
axes[1].set_xlabel("Lags (days back)")
axes[1].set_xticks(range(0, 31, 1))

plt.tight_layout()
plt.show()

The ACF and PACF plots show that the training series still contains meaningful autocorrelation. Several spikes remain outside the confidence bands, especially at lag 1 and around lag 7. This suggests that past values are informative for forecasting and that a weekly seasonal component should be considered in the SARIMAX model.

How to read these plots:

- PACF helps identify the autoregressive part of the model.
- ACF helps identify the moving average part of the model.
- A visible spike around lag 7 suggests a weekly seasonal pattern and supports the use of a seasonal component.

# 10. Prepare Exogenous Features for SARIMAX

In [ ]:
# Prepare the base dataframe for SARIMAX exogenous features
sarimax_df = pd.read_csv("../../data/processed/feature_engineered_timeseries.csv")

# Convert the date column to datetime
sarimax_df["date"] = pd.to_datetime(sarimax_df["date"])

In [ ]:
# Select the exogenous features used in the SARIMAX model
exog_cols = [
    "oil_price_interpolated",
    "is_national_holiday",
    "is_regional_holiday",
    "is_local_holiday",
    "is_weekend"
]

In [ ]:
# Split the feature-based dataframe into train and test periods
train_exog_df = sarimax_df[(sarimax_df["date"] >= "2013-01-01") & (sarimax_df["date"] <= "2013-12-31")].copy()
test_exog_df = sarimax_df[(sarimax_df["date"] >= "2014-01-01") & (sarimax_df["date"] <= "2014-03-31")].copy()

In [ ]:
# Keep only the selected exogenous columns
exog_train = train_exog_df[exog_cols].copy()
exog_test = test_exog_df[exog_cols].copy()

In [ ]:
# Check the shape of the exogenous feature sets
print("Training exog shape:", exog_train.shape)
print("Test exog shape:", exog_test.shape)

print("\nFirst 5 rows of training exogenous features:")
display(exog_train.head())

In [ ]:
# Verify that the target series and exogenous features have matching lengths
print("Train target length:", len(train))
print("Train exog length:", len(exog_train))

print("Test target length:", len(test))
print("Test exog length:", len(exog_test))

The exogenous feature dataframe does not align with the target training series, because it was created after dropping rows with missing lag and rolling features. For SARIMAX, the exogenous variables must match the target series row by row, so a separate aligned exogenous dataframe will be prepared next.

# 11. First Statistical Model

In this section, I train the first statistical forecasting model using the training period and evaluate its predictions on the test period.

In [ ]:
# Train the first SARIMAX model
model_1 = SARIMAX(
    train["unit_sales"],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 7)
)

model_1_fit = model_1.fit(disp=False)

In [ ]:
# Display the summary of the fitted SARIMAX model
print(model_1_fit.summary())

In [ ]:
# The predict() approach did not match the fitted model index for out-of-sample forecasting.
# A forecast based on the number of test steps is used instead.

In [ ]:
# Generate out-of-sample forecasts for the full test period
forecast_1 = model_1_fit.forecast(steps=len(test))

In [ ]:
forecast_1.head()

In [ ]:
forecast_1.index = test.index

In [ ]:
# Display the forecast again after assigning the test date index
forecast_1.head()

In [ ]:
# Plot actual vs forecasted values for the test period
plt.figure(figsize=(14, 5))
plt.plot(test.index, test["unit_sales"], label="Actual", color="steelblue")
plt.plot(forecast_1.index, forecast_1, label="Forecast", color="darkorange")
plt.title("Actual vs Forecasted Unit Sales")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()